# 🛡️ MathSentry — Fine-Tuning Olympiad AI on Google Colab (Free T4 GPU)

This notebook fine-tunes an open-source mathematical LLM (**Qwen 2.5 Math 7B** or **Llama 3.2 3B**) on the **MathSentry Romanian Olympiad Proof Verification Dataset** using **Unsloth (QLoRA)**.

### ⚡ Hardware Requirements:
- **Google Colab Free Tier (T4 GPU)** with 15GB VRAM.
- Training time: **~8–12 minutes**.

> **Important:** Go to `Runtime` -> `Change runtime type` -> select `T4 GPU` before running.

## Step 1: Install Unsloth and Dependencies

In [ ]:
# Install Unsloth for 5x faster training with 70% less memory on Colab T4
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes datasets

## Step 2: Load Model and Tokenizer (4-bit Quantized)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Supports long Olympiad proofs
dtype = None  # None for auto detection (Float16 on T4)
load_in_4bit = True  # Reduces memory to fit in T4 GPU (6GB VRAM)

# Select model: Qwen2.5-Math-7B-Instruct or Llama-3.2-3B-Instruct
model_name = "unsloth/Qwen2.5-Math-7B-Instruct-bnb-4bit"
# Alternative for ultra-fast training: "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Setup LoRA adapters for parameter-efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    max_seq_length=max_seq_length,
)
print("✓ Model and LoRA adapters loaded successfully!")

## Step 3: Fetch MathSentry Dataset from GitHub

In [ ]:
# Clone or pull latest MathSentry repository
import os
if not os.path.exists("MathSentry"):
    !git clone https://github.com/andr3eee1/MathSentry.git
else:
    !git -C MathSentry pull

# Build the dataset dynamically
!python3 MathSentry/dataset/generate_dataset.py

from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "MathSentry/dataset/train.jsonl",
    "validation": "MathSentry/dataset/val.jsonl"
})

print(f"✓ Loaded {len(dataset['train'])} training samples and {len(dataset['validation'])} validation samples!")

## Step 4: Format Conversations with Chat Template

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",  # Standard ChatML format
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

train_dataset = dataset["train"].map(formatting_prompts_func, batched=True)
val_dataset = dataset["validation"].map(formatting_prompts_func, batched=True)

print("Sample formatted prompt preview:")
print(train_dataset[0]["text"][:600] + "...")

## Step 5: Configure and Run SFT Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,  # ~3-4 epochs over dataset
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="mathsentry_outputs",
    ),
)

# Start training
trainer_stats = trainer.train()
print("✓ Training completed successfully!")

## Step 6: Test the Fine-Tuned Model on a Flawed Olympiad Proof

In [ ]:
# Enable native fast inference
FastLanguageModel.for_inference(model)

test_messages = [
    {
        "role": "system",
        "content": "Ești MathSentry, un motor AI de nivel olimpic specializat în verificarea formală și notarea pe barem național (ONM/OJM) a demonstrațiilor matematice scrise de mână."
    },
    {
        "role": "user",
        "content": """Evaluează următoarea demonstrație pentru problema: Fie x > -1 și n >= 1. Să se demonstreze (1+x)^n >= 1 + nx.

Rezolvare elev:
[Pas 1] Pentru n = 1: (1+x)^1 = 1+x >= 1+x, adevărat.
[Pas 2] Presupunem adevărat (1+x)^k >= 1 + kx.
[Pas 3] Înmulțim cu (1+x): (1+x)^(k+1) >= (1+kx)(1+x) = 1 + (k+1)x + kx^2.
[Pas 4] Cum kx^2 >= 0, rezultă (1+x)^(k+1) >= 1 + (k+1)x. Q.E.D."""
    }
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True)
response = tokenizer.batch_decode(outputs)
print(response[0].split("<|im_start|>assistant")[-1])

## Step 7: Save LoRA Adapter & Export

You can save the trained model locally, push to HuggingFace Hub, or export as GGUF for Ollama.

In [ ]:
# Save trained LoRA adapter
model.save_pretrained("mathsentry_lora_model")
tokenizer.save_pretrained("mathsentry_lora_model")
print("✓ Saved LoRA adapter to 'mathsentry_lora_model' folder!")

# To download to your computer from Colab:
!zip -r mathsentry_lora_model.zip mathsentry_lora_model
from google.colab import files
files.download("mathsentry_lora_model.zip")